In [2]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

opts = PdfPipelineOptions()
opts.do_ocr = False

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)}
)

doc = converter.convert("../data/text_files/Acme.pdf").document
text = doc.export_to_markdown()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 8346.40it/s]
W0816 22:37:11.416000 14612 torch/_dynamo/variables/tensor.py:1759] [0/0] Graph break from `Tensor.item()`, consider setting:
W0816 22:37:11.416000 14612 torch/_dynamo/variables/tensor.py:1759] [0/0]     torch._dynamo.config.capture_scalar_outputs = True
W0816 22:37:11.416000 14612 torch/_dynamo/variables/tensor.py:1759] [0/0] or:
W0816 22:37:11.416000 14612 torch/_dynamo/variables/tensor.py:1759] [0/0]     env TORCHDYNAMO_CAPTURE_SCALAR_OUTPUTS=1
W0816 22:37:11.416000 14612 torch/_dynamo/variables/tensor.py:1759] [0/0] to include these operations in the captured graph.
W0816 22:37:11.416000 14612 torch/_dynamo/variables/tensor.py:1759] [0/0] 
W0816 22:37:11.416000 14612 torch/_dynamo/variables/tensor.py:1759] [0/0] Graph break: from user code at:
W0816 22:37:11.416000 14612 torch/_dynamo/variables/tensor.py:1759] [0/0]   File "/Users/ishitarastogi/Docum

In [4]:
print(text)
print(doc.export_to_text())       # string, no markers at all
print(doc.export_to_dict())       # the full tree as nested dicts, nothing lost

<!-- image -->

## Acme Corp Q1 Results

Revenue grew this quarter due to strong demand in Asia.

## Sales by Region

| Region   | Q1   | Q2   |
|----------|------|------|
| Asia     | $50M | $30M |
| Europe   | $20M | $22M |
Acme Corp Q1 Results

Revenue grew this quarter due to strong demand in Asia.

Sales by Region

| Region   | Q1   | Q2   |
|----------|------|------|
| Asia     | $50M | $30M |
| Europe   | $20M | $22M |
{'schema_name': 'DoclingDocument', 'version': '1.10.0', 'name': 'Acme', 'origin': {'mimetype': 'application/pdf', 'binary_hash': 8338287423662425922, 'filename': 'Acme.pdf'}, 'furniture': {'self_ref': '#/furniture', 'children': [], 'content_layer': 'furniture', 'name': '_root_', 'label': 'unspecified'}, 'body': {'self_ref': '#/body', 'children': [{'$ref': '#/pictures/0'}, {'$ref': '#/texts/0'}, {'$ref': '#/texts/1'}, {'$ref': '#/texts/2'}, {'$ref': '#/tables/0'}], 'content_layer': 'body', 'name': '_root_', 'label': 'unspecified'}, 'groups': [], 'texts': [{'self_re

In [ ]:
text = doc.export_to_markdown()

chunks = text.split("\n#")           # cut wherever a heading starts
chunks = [c.strip() for c in chunks if c.strip()]

print(len(chunks), "chunks")
print(chunks[0])

['<!-- image -->\n', '# Acme Corp Q1 Results\n\nRevenue grew this quarter due to strong demand in Asia.\n', '# Sales by Region\n\n| Region   | Q1   | Q2   |\n|----------|------|------|\n| Asia     | $50M | $30M |\n| Europe   | $20M | $22M |']
['<!-- image -->', '# Acme Corp Q1 Results\n\nRevenue grew this quarter due to strong demand in Asia.', '# Sales by Region\n\n| Region   | Q1   | Q2   |\n|----------|------|------|\n| Asia     | $50M | $30M |\n| Europe   | $20M | $22M |']
3 chunks
# Sales by Region

| Region   | Q1   | Q2   |
|----------|------|------|
| Asia     | $50M | $30M |
| Europe   | $20M | $22M |


In [11]:
def chunk(text, max_chars=2000):
    sections = [s for s in text.split("\n#") if s.strip()]
    out = []
    for s in sections:
        if len(s) <= max_chars:
            out.append(s.strip())
        else:
            # section too big — split it on blank lines instead
            para, buf = s.split("\n\n"), ""
            for p in para:
                if len(buf) + len(p) > max_chars:
                    out.append(buf.strip())
                    buf = p
                else:
                    buf += "\n\n" + p
            if buf.strip():
                out.append(buf.strip())
    return out


chunks = chunk(text)
print(len(chunks), "chunks")
print(chunks[0])

3 chunks
<!-- image -->


In [14]:
from docling.chunking import HybridChunker

chunker = HybridChunker(max_tokens=512)
chunks = [c.text for c in chunker.chunk(doc)]
print(len(chunks), "chunks")
print(chunks)

2 chunks
['Revenue grew this quarter due to strong demand in Asia.', 'Region, 1 = Q1. Region, 2 = Q2. Asia, 1 = $50M. Asia, 2 = $30M. Europe, 1 = $20M. Europe, 2 = $22M']
